Compute model evidence, P(D|M), for all models developped in this project.

FEV1, 2-day FEV1 FEF, long model

In [1]:
import concurrent.futures
from itertools import repeat

import numpy as np
import pandas as pd

import data.breathe_data as bd
import data.helpers as dh
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change as cca_ar_change
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change_noo2sat as cca_ar_change_noo2sat
import model_validation.model_evidence as me

In [2]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


In [3]:
# P(D|M) on the strict 30-day long model (excluding ID below 30 days)
# Without FEF25-75, log_p_S_given_D = -49.08053225
# With FEV1 and FEF25-75, log_p_S_given_D = -148.65322444

In [3]:
# Reduce dataset to the last 30-day sequences

ndays = 20
df20 = pd.DataFrame(columns=df.columns)
for id in df.ID.unique():
    df_pre, start_idx, end_idx = dh.find_longest_conseq_sequence(
        df[df.ID == id], n_missing_days_allowed=1
    )

    dftmp = df_pre.tail(ndays).reset_index()

    if len(dftmp) < ndays:
        # print(f"Skipping ID {id}, n entries < {ndays} days")
        continue

    df20 = pd.concat([df20, dftmp])

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_40424/546156911.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df20 = pd.concat([df20, dftmp])


In [4]:
df20.ID.nunique()

83

# Longitudinal model

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_long_model = list(
            executor.map(me.process_id_long_model, df20.ID.unique(), repeat(df20))
        )

101 - Time for 20 entries: 5.61 s
[-93.56807729406881]


# 2 day FEV1, FEF25-75 model


I had to update existing implementation, made for the long model with interconnected AR. To do so, I set the AR vevidence to the first_day_prior each time. 

When fusing the models, I only took P(FEV1 day n|M, rmax FEV1, rmax FEF2575) * P(FEF2575 day n|M, rmax FEV1, rmax FEF2575, FEV1 day n). This allowed to compare the results with the other models

In [5]:
df_rmax_rows = (
    df.sort_values(by=["ecFEV1", "ecFEF2575", "O2 Saturation"], ascending=False)
    .groupby("ID")
    .agg(lambda df: df.head(1))
    .reset_index()
)
df_rmax_rows = df_rmax_rows[df_rmax_rows.ID.isin(df20.ID.unique())]

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_2day_fev1_fef_model = list(
            executor.map(me.process_id_2day_fev1_fef_model, df20.ID.unique(), repeat(df20), repeat(df_rmax_rows))
        )


101 - Time for 2 entries: 1.28 s
102 - Time for 2 entries: 1.34 s
101 - Time for 2 entries: 1.03 s
102 - Time for 2 entries: 1.00 s
102 - Time for 2 entries: 0.93 s
101 - Time for 2 entries: 1.04 s
102 - Time for 2 entries: 0.98 s
101 - Time for 2 entries: 1.07 s
102 - Time for 2 entries: 1.07 s
101 - Time for 2 entries: 0.94 s
101 - Time for 2 entries: 1.01 s
102 - Time for 2 entries: 1.15 s
102 - Time for 2 entries: 1.31 s
101 - Time for 2 entries: 1.42 s
102 - Time for 2 entries: 1.11 s
101 - Time for 2 entries: 1.18 s
102 - Time for 2 entries: 1.08 s
101 - Time for 2 entries: 1.15 s
102 - Time for 2 entries: 1.06 s
101 - Time for 2 entries: 1.07 s
102 - Time for 2 entries: 1.22 s
101 - Time for 2 entries: 1.12 s
102 - Time for 2 entries: 0.94 s
101 - Time for 2 entries: 1.01 s
102 - Time for 2 entries: 0.99 s
101 - Time for 2 entries: 0.91 s
102 - Time for 2 entries: 0.92 s
101 - Time for 2 entries: 1.05 s
102 - Time for 2 entries: 0.97 s
101 - Time for 2 entries: 0.97 s
102 - Time

In [11]:
res_2day_fev1_fef_model

[-138.37374487152357, -146.22196160048034]

In [ ]:
# with rmax FEV1, 101: -276.9759478315757
# with rmax but just taking the curr day probs, 101: -138

# FEV1, FEF25-75 model

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_fev1_fef_model = list(
            executor.map(
                # me.process_id_fev1_fef_model, ['101'], repeat(df20)
                me.process_id_fev1_fef_model, df20.ID.unique(), repeat(df20)
            )
        )

101 - Time for 1 entries: 0.49 s
101 - Time for 1 entries: 0.48 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.91 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.51 s
101 - Time for 1 entries: 0.49 s
101 - Time for 1 entries: 0.52 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.62 s
101 - Time for 1 entries: 0.50 s
101 - Time for 1 entries: 0.44 s
101 - Time for 1 entries: 0.46 s
101 - Time for 1 entries: 0.47 s
101 - Time for 1 entries: 0.46 s
101 - Time for 1 entries: 0.46 s
101 - Time for 1 entries: 0.47 s


In [12]:
np.sum(results)

-149.23177954891864